# Liu2024 — Cross-Subject S-JEPA × Riemannian Fusion (Euclidean-Aligned, Leave-One-Subject-Out)

**The idea.** The decodability analysis showed S-JEPA is stuck at chance mainly because
within-subject it only ever sees ~32 training trials. This notebook attacks that directly:

1. **Euclidean Alignment (EA)** — whiten each subject's trials by their reference covariance
   `R⁻¹ᐟ²`, so different subjects' data become statistically comparable (He & Wu, 2020).
2. **Two branches on the aligned signal** — (a) frozen pretrained **S-JEPA embeddings**, and
   (b) **Riemannian tangent-space** features from the aligned covariances.
3. **Fusion** — standardize each block and concatenate into one **regularized linear head**
   (logistic regression), with an optional probability-level **stacking** variant.
4. **Leave-One-Subject-Out** — train on the pooled aligned trials of all other subjects and test
   on the held-out one. Now the model trains on hundreds–thousands of trials instead of 32.
5. **Evaluate on the decodable subgroup** from the permutation notebook (configurable).

**Why this can reach the ~70% range when within-subject S-JEPA can't:** EA + cross-subject
pooling is the standard way deep EEG models get enough data to generalize, and fusing the
physiology-matched Riemannian features gives the linear head a strong, ready-made signal to lean
on while the S-JEPA embedding contributes whatever complementary structure it has captured.

> **Honesty.** 70% is a *hypothesis*, not a guarantee — it depends on your data. The EA +
> Riemannian + fusion backbone is fully validated here; the S-JEPA branch uses the carried
> pretrained encoder and may need an environment-specific tweak to the embedding layer name
> (`CONFIG["fusion"]["embedding_module"]`), or you can feed precomputed embeddings. If the
> encoder isn't available the notebook runs Riemann-only and says so.

# 1. Setup

In [1]:
import os, re, json, hashlib, random, builtins, platform, inspect
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy import signal
from scipy.linalg import eigh

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import sys

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False; print(f"[setup] matplotlib unavailable: {exc}")

try:
    import torch
    HAVE_TORCH = True
except Exception as exc:
    HAVE_TORCH = False; print(f"[setup] torch unavailable -> S-JEPA branch needs precomputed embeddings: {exc}")

from torch.utils.data import Dataset, Subset, DataLoader

try:
    import mne; mne.set_log_level("WARNING"); HAVE_MNE = True
except Exception as exc:
    HAVE_MNE = False; print(f"[setup] mne unavailable -> data loading skipped: {exc}")

try:
    from braindecode.models import SignalJEPA_PreLocal
    HAVE_BRAINDECODE = True
except Exception as exc:
    HAVE_BRAINDECODE = False; print(f"[setup] braindecode/SignalJEPA unavailable -> S-JEPA branch via precomputed embeddings only: {exc}")

try:
    from pyriemann.estimation import Covariances
    from pyriemann.tangentspace import TangentSpace
    HAVE_PYRIEMANN = True
except Exception as exc:
    HAVE_PYRIEMANN = False; print(f"[setup] pyriemann unavailable -> log-Euclidean tangent fallback: {exc}")

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("deps:", dict(torch=HAVE_TORCH, mne=HAVE_MNE, braindecode=HAVE_BRAINDECODE,
                    pyriemann=HAVE_PYRIEMANN, mpl=HAVE_MPL))


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


deps: {'torch': True, 'mne': True, 'braindecode': True, 'pyriemann': True, 'mpl': True}


# 2. Configuration

## 2.1 Channels (carried)

In [2]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


## 2.2 CONFIG (carried) — S-JEPA window/rate kept as the pretrained encoder expects

In [3]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "baseline_sjepa_prelocal",
    "config_note": "Clean MNE-style preprocessing pipeline builder + S-JEPA hyperparameter controls.",

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ------------------------------------------------------------------
    # Source-domain preprocessing before creating MNE RawArray
    # ------------------------------------------------------------------
    "demean_mode": "none",      # none, trial_mean, baseline_window_mean
    "baseline_window_s": [0.0, 2.0],
    "detrend_mode": "none",                     # none, constant, linear
    "eog_correction": "none",                   # none, linear_regression

    # Robust source-domain clipping / winsorization. Use cautiously.
    "artifact_clip_mode": "none",               # none, absolute, percentile
    "artifact_clip_abs_value": None,             # in source_unit, e.g. 150.0 when source_unit=microvolts
    "artifact_clip_percentile": 99.5,

    # ------------------------------------------------------------------
    # MNE Raw-level preprocessing
    # ------------------------------------------------------------------
    "reference_mode": "average",                # average, none
    "reference_timing": "before_resample_filter",  # before_resample_filter, after_resample_before_filter, after_filter
    "resample": True,
    "resample_sfreq": 128,

    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",                     # fir, iir
    "filter_phase": "zero",                     # zero, zero-double, minimum (FIR only)
    "filter_fir_design": "firwin",              # firwin, firwin2 (FIR only)
    "filter_l_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_h_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_iir_params": None,                    # example: {"order": 2, "ftype": "butter"}

    "notch_freqs": None,                          # example: [50.0]
    "notch_before_bandpass": False,

    # ------------------------------------------------------------------
    # Windowing and post-window cleaning
    # ------------------------------------------------------------------
    "target_window_s": 4.2,
    "target_window_samples": 537,
    "mi_window_start_s": 1.5,

    "reject_bad_trials": False,
    "reject_peak_to_peak_threshold": None,       # in final_model_unit
    "reject_abs_threshold": None,                # in final_model_unit
    "min_trials_per_class_after_reject": None,

    # ------------------------------------------------------------------
    # Fold-safe normalization. train_* modes are fit on each training split only.
    # ------------------------------------------------------------------
    "normalization_mode": "none",               # none, train_global_zscore, train_channel_zscore, train_channel_robust, trial_global_zscore, trial_channel_zscore
    "normalization_eps": 1e-6,

    # ------------------------------------------------------------------
    # Model / downstream strategy
    # ------------------------------------------------------------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",                          # new, full
    "warmup_epochs": 10,

    # ------------------------------------------------------------------
    # Evaluation protocol
    # ------------------------------------------------------------------
    "evaluation_mode": "stratified_kfold",      # stratified_kfold, liu2024_repeated_60_40, repeated_stratified_split
    "cv_folds": 5,
    "n_repeats": 10,
    "test_size": 0.4,
    "split_random_state": 2026,
    "assert_balanced_folds": True,

    # ------------------------------------------------------------------
    # Training hyperparameters
    # ------------------------------------------------------------------
    "batch_size": 4,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "val_split": 0.2,
    "learning_rate": 0.0003,
    "optimizer_name": "adam",                   # adam, adamw
    "weight_decay": 0.0,
    "gradient_clip_norm": None,
    "checkpoint_metric": "valid_loss",           # valid_loss, valid_balanced_accuracy
    "label_smoothing": 0.0,
    "prediction_balance_loss_weight": 1.0,

    # ------------------------------------------------------------------
    # braindecode on-the-fly augmentation.
    # Applied to the TRAINING iterator ONLY (via AugmentedDataLoader), so the
    # validation split skorch carves out internally is never augmented -> no leakage.
    # There is no fixed "number of augmented samples": the model sees a freshly
    # augmented view of the train fold every epoch. Control INTENSITY with each
    # transform's "probability" (how often it fires) and its magnitude params.
    # Ready-to-use configs are in the markdown cell just below CONFIG.
    # ------------------------------------------------------------------
    # "augmentation": {
    #     "enabled": False,        # master switch
    #     "name": "none",          # label, saved with artifacts
    #     "random_state": 2026,
    #     # each entry: {"name": <transform>, "probability": 0..1, <transform params>}
    #     "transforms": [],
    # },

    "augmentation": {
        "enabled": True,
        "name": "time_mask",
        "random_state": 2026,
        "transforms": [
        {
            "mask_len_samples": 64,
            "name": "smooth_time_mask",
            "probability": 0.5
        }
        ]
    },

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
    "val_split_random_state": 2026,

    # ------------------------------------------------------------------
    # Diagnostics / interpretation
    # ------------------------------------------------------------------
    "extract_spatial_conv_weights": True,
    "save_spatial_weight_plots": False,
    "plot_individual_spatial_filters": False,
    "max_spatial_filters_to_plot": 8,
    "topomap_dpi": 160,
    "topomap_value_mode": "relative_zscore",    # raw, relative_zscore, relative_percent
    "topomap_cmap": "RdBu_r",
    "collapse_threshold": 0.9,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

## 2.3 Fusion / cross-subject settings

In [4]:
CONFIG["experiment_name"] = "sjepa_riemann_crosssubject_fusion"
CONFIG["artifact_dir"] = str(WORKING_DIR / "artifacts" / "liu2024-sjepa-riemann-crosssubject-fusion")
CONFIG["config_note"] = "EA + cross-subject LOSO; fuse frozen S-JEPA embeddings with Riemannian tangent features."
CONFIG["augmentation"] = {"enabled": False, "name": "none", "random_state": 2026, "transforms": []}

CONFIG["fusion"] = {
    "eval_mode": "loso",                 # 'loso' (cross-subject) or 'within' (per-subject CV)
    "decodable_subjects": [7, 20, 23, 26, 28, 37, 38, 40, 44, 45],  # from the permutation notebook; or "all"
    "branches": ["riemann", "sjepa", "fusion"],   # which to evaluate/compare
    "fusion_method": "concat_logreg",    # 'concat_logreg' or 'stacking'
    "euclidean_alignment": True,
    "align_before_sjepa": True,          # feed EA-aligned signal to the encoder (cross-subject comparable)
    "tangent_metric": "riemann",
    "cov_estimator": "oas",
    "logreg_C": 1.0,
    "standardize": True,
    "embedding_module": "final_layer",   # capture the INPUT to this submodule as the embedding
    "embedding_pool": "mean",            # 'mean' over tokens if 3D, else 'flatten'
    "embedding_batch": 64,
    "sjepa_embeddings_path": None,       # optional .npz with arrays X (n,d) aligned to X_ALL order
    "within_cv_folds": 5,
    "seed": 2026,
}
print("Fusion config:", {k: CONFIG["fusion"][k] for k in
      ["eval_mode", "branches", "fusion_method", "euclidean_alignment", "tangent_metric"]})
print("Decodable subjects:", CONFIG["fusion"]["decodable_subjects"])


Fusion config: {'eval_mode': 'loso', 'branches': ['riemann', 'sjepa', 'fusion'], 'fusion_method': 'concat_logreg', 'euclidean_alignment': True, 'tangent_metric': 'riemann'}
Decodable subjects: [7, 20, 23, 26, 28, 37, 38, 40, 44, 45]


## 2.4 Constants / artifacts / reproducibility (carried)

In [5]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels.
# Keep the 29 EEG channels used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2

if bool(CONFIG.get("resample", True)):
    EFFECTIVE_SFREQ = float(CONFIG.get("resample_sfreq", 128))
else:
    EFFECTIVE_SFREQ = float(LIU_SOURCE_SFREQ)

CONFIG["effective_sfreq"] = EFFECTIVE_SFREQ
CONFIG["sfreq"] = EFFECTIVE_SFREQ  # compatibility with existing cells/artifacts

if CONFIG.get("target_window_samples", None) is None:
    WINDOW_SAMPLES = int(round(float(CONFIG["target_window_s"]) * EFFECTIVE_SFREQ))
else:
    WINDOW_SAMPLES = int(CONFIG["target_window_samples"])

TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * EFFECTIVE_SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

PREPROCESSING_KEYS = [
    "source_unit", "final_model_unit",
    "demean_mode", "baseline_window_s", "detrend_mode", "eog_correction",
    "artifact_clip_mode", "artifact_clip_abs_value", "artifact_clip_percentile",
    "reference_mode", "reference_timing", "resample", "resample_sfreq", "effective_sfreq",
    "filter_enabled", "filter_low", "filter_high", "filter_method", "filter_phase",
    "filter_fir_design", "filter_l_trans_bandwidth", "filter_h_trans_bandwidth", "filter_iir_params",
    "notch_freqs", "notch_before_bandpass",
    "mi_window_start_s", "target_window_s", "target_window_samples",
    "reject_bad_trials", "reject_peak_to_peak_threshold", "reject_abs_threshold",
    "normalization_mode", "normalization_eps",
]

TRAINING_KEYS = [
    "strategy", "batch_size", "learning_rate", "optimizer_name", "weight_decay",
    "val_split", "early_stopping_patience", "n_epochs",
    "augmentation",
]

EVALUATION_KEYS = [
    "evaluation_mode", "cv_folds", "n_repeats", "test_size",
    "cv_random_state", "split_random_state", "val_split_random_state",
]

def summarize_selected_config(keys):
    return {k: CONFIG.get(k) for k in keys}

PREPROCESSING_CONFIG = summarize_selected_config(PREPROCESSING_KEYS)
TRAINING_CONFIG = summarize_selected_config(TRAINING_KEYS)
EVALUATION_CONFIG = summarize_selected_config(EVALUATION_KEYS)

def print_config_block(title, values):
    print(title)
    for key, value in values.items():
        print(f"  {key:34s}: {value}")

print("Effective Liu2024 Source MAT settings:")
print(f"  Experiment:                        {CONFIG.get('experiment_name')}")
print(f"  Note:                              {CONFIG.get('config_note')}")
print(f"  Channels:                          {len(EEG_CHANNEL_NAMES)}")
print(f"  Channel names:                     {EEG_CHANNEL_NAMES}")
print(f"  Source sfreq:                      {LIU_SOURCE_SFREQ} Hz")
print(f"  Effective sfreq:                   {EFFECTIVE_SFREQ} Hz")
print(f"  MI window start / samples:         {CONFIG['mi_window_start_s']} s / {WINDOW_SAMPLES}")
print(f"  Effective window duration:         {TARGET_TRIAL_DURATION_S:.4f} s")
print(f"  Evaluation mode:                   {CONFIG.get('evaluation_mode')}")
print(f"  Fixed seed:                        base={CONFIG.get('seed')} | cv={CONFIG.get('cv_random_state')} | split={CONFIG.get('split_random_state')} | val={CONFIG.get('val_split_random_state')}")
print_config_block("\nPreprocessing config:", PREPROCESSING_CONFIG)
print_config_block("\nTraining config:", TRAINING_CONFIG)
print_config_block("\nEvaluation config:", EVALUATION_CONFIG)


Effective Liu2024 Source MAT settings:
  Experiment:                        sjepa_riemann_crosssubject_fusion
  Note:                              EA + cross-subject LOSO; fuse frozen S-JEPA embeddings with Riemannian tangent features.
  Channels:                          29
  Channel names:                     ['Fp1', 'Fp2', 'Fz', 'F3', 'F4', 'F7', 'F8', 'FCz', 'FC3', 'FC4', 'FT7', 'FT8', 'Cz', 'C3', 'C4', 'T3', 'T4', 'CP3', 'CP4', 'TP7', 'TP8', 'Pz', 'P3', 'P4', 'T5', 'T6', 'Oz', 'O1', 'O2']
  Source sfreq:                      500 Hz
  Effective sfreq:                   128.0 Hz
  MI window start / samples:         1.5 s / 537
  Effective window duration:         4.1953 s
  Evaluation mode:                   stratified_kfold
  Fixed seed:                        base=2026 | cv=2026 | split=2026 | val=2026

Preprocessing config:
  source_unit                       : microvolts
  final_model_unit                  : microvolts
  demean_mode                       : none
  baseline_window

In [6]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


[2026-06-14 11:20:32] Run ID:     20260614_1120_82d1a0d1
[2026-06-14 11:20:32] Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-sjepa-riemann-crosssubject-fusion/20260614_1120_82d1a0d1
[2026-06-14 11:20:32] Config:     /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-sjepa-riemann-crosssubject-fusion/20260614_1120_82d1a0d1/config.json


In [7]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


[2026-06-14 11:20:32] Using device: mps
[2026-06-14 11:20:33] Seed initialized: 2026


# 3. Carried machinery (verbatim)

### 3.1 Data-loading helpers

In [8]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


### 3.2 Preprocessing pipeline

In [9]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES),  # type: ignore
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def source_values_to_mne_volts(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("source_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e-6
    if unit in ("mv", "millivolt", "millivolts"):
        return arr * 1e-3
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported source_unit={config.get('source_unit')}")

def mne_volts_to_model_unit(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("final_model_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e6
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported final_model_unit={config.get('final_model_unit')}")

def _none_like(value):
    return value is None or str(value).lower() in ("none", "off", "false", "")

def build_preprocessing_pipeline(config):
    """Return a readable pipeline plan.

    The pipeline is represented as a list of dictionaries rather than hidden global logic.
    Every row is logged and saved in the run metadata through the preprocessing step list.
    """
    pipeline = []

    # Fixed source structure.
    pipeline.append({
        "stage": "source",
        "name": "select_eeg_channels",
        "description": "select Liu EEG channels, drop CPz reference, EOG, and marker before model input",
        "enabled": True,
    })

    pipeline.append({
        "stage": "source",
        "name": "demean",
        "mode": config.get("demean_mode", "none"),
        "baseline_window_s": config.get("baseline_window_s"),
        "enabled": not _none_like(config.get("demean_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "detrend",
        "mode": config.get("detrend_mode", "none"),
        "enabled": not _none_like(config.get("detrend_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "eog_correction",
        "mode": config.get("eog_correction", "none"),
        "enabled": not _none_like(config.get("eog_correction", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "artifact_clipping",
        "mode": config.get("artifact_clip_mode", "none"),
        "abs_value": config.get("artifact_clip_abs_value"),
        "percentile": config.get("artifact_clip_percentile"),
        "enabled": not _none_like(config.get("artifact_clip_mode", "none")),
    })

    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()
    if reference_timing == "before_resample_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "before_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({"stage": "mne_raw", "name": "resample", "sfreq": config.get("resample_sfreq"), "enabled": bool(config.get("resample", True))})

    if reference_timing == "after_resample_before_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    pipeline.append({
        "stage": "mne_raw",
        "name": "bandpass_filter",
        "enabled": bool(config.get("filter_enabled", True)),
        "l_freq": config.get("filter_low"),
        "h_freq": config.get("filter_high"),
        "method": config.get("filter_method"),
        "phase": config.get("filter_phase"),
        "fir_design": config.get("filter_fir_design"),
        "iir_params": config.get("filter_iir_params"),
    })

    if reference_timing == "after_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if not bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "after_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({
        "stage": "window",
        "name": "crop_fixed_mi_window",
        "start_s": config.get("mi_window_start_s"),
        "target_window_samples": config.get("target_window_samples"),
        "enabled": True,
    })

    pipeline.append({
        "stage": "window",
        "name": "bad_trial_rejection",
        "enabled": bool(config.get("reject_bad_trials", False)),
        "peak_to_peak_threshold": config.get("reject_peak_to_peak_threshold"),
        "abs_threshold": config.get("reject_abs_threshold"),
    })

    pipeline.append({
        "stage": "split",
        "name": "fold_safe_normalization",
        "mode": config.get("normalization_mode", "none"),
        "enabled": not _none_like(config.get("normalization_mode", "none")),
    })

    return pipeline

def describe_pipeline(pipeline):
    lines = []
    for step in pipeline:
        status = "ON" if step.get("enabled", False) else "off"
        parts = [f"[{status}] {step.get('stage')}::{step.get('name')}"]
        for key, value in step.items():
            if key not in ("stage", "name", "description", "enabled") and value is not None:
                parts.append(f"{key}={value}")
        if step.get("description"):
            parts.append(f"- {step['description']}")
        lines.append(" | ".join(parts))
    return lines

PREPROCESSING_PIPELINE = build_preprocessing_pipeline(CONFIG)
print("Configured preprocessing pipeline:")
for line in describe_pipeline(PREPROCESSING_PIPELINE):
    print("  - " + line)

def apply_source_demean(X_eeg, subject_id, config, steps):
    mode = str(config.get("demean_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source demean skipped")
        return X

    if mode == "trial_mean":
        steps.append("source demean: subtract trial/channel mean over time")
        return X - X.mean(axis=-1, keepdims=True)

    if mode == "baseline_window_mean":
        baseline = config.get("baseline_window_s", [0.0, 2.0])
        if baseline is None or len(baseline) != 2:
            raise ValueError("baseline_window_s must be [start_s, stop_s] for baseline_window_mean.")
        start_s, stop_s = float(baseline[0]), float(baseline[1])
        start = int(round(start_s * LIU_SOURCE_SFREQ))
        stop = int(round(stop_s * LIU_SOURCE_SFREQ))
        if start < 0 or stop <= start or stop > X.shape[-1]:
            raise ValueError(f"Subject {subject_id}: invalid baseline_window_s={baseline} for source length {X.shape[-1]}")
        steps.append(f"source demean: subtract baseline mean {baseline}s")
        return X - X[:, :, start:stop].mean(axis=-1, keepdims=True)

    raise ValueError(f"Unsupported demean_mode={config.get('demean_mode')}")

def apply_source_detrend(X_eeg, subject_id, config, steps):
    mode = str(config.get("detrend_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)
    if mode in ("none", "off", "false"):
        steps.append("source detrend skipped")
        return X
    if mode == "constant":
        steps.append("source detrend: scipy.signal.detrend(type='constant')")
        return signal.detrend(X, axis=-1, type="constant")
    if mode == "linear":
        steps.append("source detrend: scipy.signal.detrend(type='linear')")
        return signal.detrend(X, axis=-1, type="linear")
    raise ValueError(f"Unsupported detrend_mode={config.get('detrend_mode')}")

def apply_eog_correction(X_eeg, rawdata, subject_id, config, steps):
    mode = str(config.get("eog_correction", "none")).lower()
    if mode in ("none", "off", "false"):
        steps.append("EOG correction skipped")
        return X_eeg

    if mode != "linear_regression":
        raise ValueError(
            "Only eog_correction='linear_regression' is implemented in this source-MAT notebook. "
            "ICA is intentionally not included because the source pipeline drops EOG before model input and "
            "the dataset has only 40 trials per subject."
        )

    if rawdata.shape[1] <= max(SOURCE_EOG_CHANNEL_INDICES):
        raise ValueError(f"Subject {subject_id}: rawdata does not contain expected EOG channels.")

    X = np.asarray(X_eeg, dtype=np.float64)
    eog = np.asarray(rawdata[:, SOURCE_EOG_CHANNEL_INDICES, :], dtype=np.float64)

    n_trials, n_chans, n_samples = X.shape
    eog_2d = eog.transpose(0, 2, 1).reshape(-1, len(SOURCE_EOG_CHANNEL_INDICES))
    eeg_2d = X.transpose(0, 2, 1).reshape(-1, n_chans)

    design = np.column_stack([np.ones(eog_2d.shape[0]), eog_2d])
    beta, *_ = np.linalg.lstsq(design, eeg_2d, rcond=None)
    eog_contribution = design[:, 1:] @ beta[1:, :]
    corrected = eeg_2d - eog_contribution
    corrected = corrected.reshape(n_trials, n_samples, n_chans).transpose(0, 2, 1)

    steps.append("EOG correction: linear regression using HEOG/VEOG before dropping EOG")
    return corrected

def apply_source_artifact_clipping(X_eeg, subject_id, config, steps):
    mode = str(config.get("artifact_clip_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source artifact clipping skipped")
        return X, {"artifact_clip_applied": False, "artifact_clip_threshold": None}

    if mode == "absolute":
        threshold = config.get("artifact_clip_abs_value")
        if threshold is None:
            raise ValueError("artifact_clip_abs_value must be set when artifact_clip_mode='absolute'.")
        threshold = float(threshold)
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: absolute ±{threshold:g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    if mode == "percentile":
        pct = float(config.get("artifact_clip_percentile", 99.5))
        threshold = float(np.nanpercentile(np.abs(X), pct))
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: percentile {pct:g}% -> ±{threshold:.4g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_percentile": pct,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    raise ValueError(f"Unsupported artifact_clip_mode={config.get('artifact_clip_mode')}")

def apply_reference(raw, config, steps, timing_label):
    mode = str(config.get("reference_mode", "average")).lower()
    if mode in ("none", "off", "false"):
        steps.append(f"reference skipped at {timing_label}")
        return raw
    if mode == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
        steps.append(f"average reference at {timing_label}")
        return raw
    raise ValueError(f"Unsupported reference_mode={config.get('reference_mode')}")

def apply_notch(raw, config, steps, timing_label):
    freqs = config.get("notch_freqs", None)
    if freqs is None or freqs == []:
        return raw
    raw.notch_filter(freqs=freqs, verbose=False)
    steps.append(f"notch filter {freqs} Hz at {timing_label}")
    return raw

def apply_resample(raw, config, steps):
    if bool(config.get("resample", True)):
        target = float(config.get("resample_sfreq", 128))
        raw.resample(target, verbose=False)
        steps.append(f"resample to {target:g} Hz")
    else:
        steps.append("resample skipped; kept source 500 Hz")
    return raw

def _float_or_none_or_auto(value):
    if value is None:
        return None
    if isinstance(value, str) and value.lower() == "auto":
        return "auto"
    return float(value)

def apply_bandpass(raw, config, steps):
    if not bool(config.get("filter_enabled", True)):
        steps.append("bandpass skipped")
        return raw

    l_freq = config.get("filter_low", None)
    h_freq = config.get("filter_high", None)
    l_freq = None if l_freq is None else float(l_freq)
    h_freq = None if h_freq is None else float(h_freq)

    method = str(config.get("filter_method", "fir")).lower()
    if method == "iir":
        iir_params = config.get("filter_iir_params", None)
        if iir_params is None:
            iir_params = {"order": 2, "ftype": "butter"}
        raw.filter(l_freq=l_freq, h_freq=h_freq, method="iir", iir_params=iir_params, verbose=False)
        steps.append(f"IIR bandpass {l_freq}–{h_freq} Hz | params={iir_params}")
    elif method == "fir":
        raw.filter(
            l_freq=l_freq,
            h_freq=h_freq,
            method="fir",
            phase=config.get("filter_phase", "zero"),
            fir_design=config.get("filter_fir_design", "firwin"),
            l_trans_bandwidth=_float_or_none_or_auto(config.get("filter_l_trans_bandwidth", "auto")),
            h_trans_bandwidth=_float_or_none_or_auto(config.get("filter_h_trans_bandwidth", "auto")),
            verbose=False,
        )
        steps.append(
            f"FIR bandpass {l_freq}–{h_freq} Hz | phase={config.get('filter_phase')} | "
            f"fir_design={config.get('filter_fir_design')}"
        )
    else:
        raise ValueError(f"Unsupported filter_method={config.get('filter_method')}")
    return raw

def apply_mne_raw_pipeline(raw, config, steps):
    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()

    if reference_timing == "before_resample_filter":
        raw = apply_reference(raw, config, steps, "before resample/filter")

    if bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "before resample/filter")

    raw = apply_resample(raw, config, steps)

    if reference_timing == "after_resample_before_filter":
        raw = apply_reference(raw, config, steps, "after resample before filter")

    raw = apply_bandpass(raw, config, steps)

    if reference_timing == "after_filter":
        raw = apply_reference(raw, config, steps, "after filter")

    if not bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "after bandpass")

    return raw

def maybe_reject_bad_trials(X_win, y, subject_id, config, steps):
    stats = {
        "reject_bad_trials": bool(config.get("reject_bad_trials", False)),
        "n_trials_before_reject": int(len(y)),
        "n_trials_after_reject": int(len(y)),
        "n_rejected_trials": 0,
        "rejection_skipped": False,
    }

    if not bool(config.get("reject_bad_trials", False)):
        steps.append("bad-trial rejection skipped")
        return X_win, y, stats

    keep = np.ones(len(y), dtype=bool)

    ptp_threshold = config.get("reject_peak_to_peak_threshold", None)
    if ptp_threshold is not None:
        ptp = np.ptp(X_win, axis=-1).max(axis=1)
        keep &= ptp <= float(ptp_threshold)
        stats["reject_peak_to_peak_threshold"] = float(ptp_threshold)
        stats["max_trial_peak_to_peak"] = float(np.max(ptp))

    abs_threshold = config.get("reject_abs_threshold", None)
    if abs_threshold is not None:
        max_abs = np.max(np.abs(X_win), axis=(1, 2))
        keep &= max_abs <= float(abs_threshold)
        stats["reject_abs_threshold"] = float(abs_threshold)
        stats["max_trial_abs"] = float(np.max(max_abs))

    proposed_y = y[keep]
    min_required = config.get("min_trials_per_class_after_reject", None)
    if min_required is None:
        if config.get("evaluation_mode") == "liu2024_repeated_60_40":
            min_required = 12
        else:
            min_required = int(config.get("cv_folds", 5))
    proposed_counts = np.bincount(proposed_y, minlength=TARGET_N_CLASSES)

    if len(proposed_y) == 0 or proposed_counts.min() < int(min_required):
        steps.append(
            "bad-trial rejection skipped because it would leave too few samples "
            f"per class: proposed_counts={proposed_counts.tolist()}, min_required={min_required}"
        )
        stats["rejection_skipped"] = True
        stats["proposed_class_counts_after_reject"] = proposed_counts.tolist()
        return X_win, y, stats

    X_new = X_win[keep]
    y_new = proposed_y
    stats["n_trials_after_reject"] = int(len(y_new))
    stats["n_rejected_trials"] = int(np.sum(~keep))
    stats["class_counts_after_reject"] = np.bincount(y_new, minlength=TARGET_N_CLASSES).tolist()
    steps.append(
        f"bad-trial rejection applied: rejected={stats['n_rejected_trials']} / {stats['n_trials_before_reject']} | "
        f"class_counts={stats['class_counts_after_reject']}"
    )
    return X_new, y_new, stats

def preprocess_subject_configurable(rawdata, labels, subject_id, config=None):
    """Apply the configured preprocessing pipeline to one Liu2024 source subject.

    Order:
      1. source-domain operations: channel selection, mean removal, detrending, EOG regression, clipping
      2. MNE RawArray operations: reference, notch, resample, bandpass
      3. window crop and optional trial rejection
      4. fold-safe normalization later inside training split code
    """
    config = CONFIG if config is None else config

    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")

    n_trials = rawdata.shape[0]
    steps = describe_pipeline(build_preprocessing_pipeline(config))
    runtime_steps = []
    preprocessing_stats = {}

    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    runtime_steps.append("select 29 EEG channels; drop CPz source reference, EOG, and marker")

    X_eeg = apply_source_demean(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_source_detrend(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_eog_correction(X_eeg, rawdata, subject_id, config, runtime_steps)
    X_eeg, clip_stats = apply_source_artifact_clipping(X_eeg, subject_id, config, runtime_steps)
    preprocessing_stats.update(clip_stats)

    X_eeg_volts = source_values_to_mne_volts(X_eeg, config)
    runtime_steps.append(f"convert source {config.get('source_unit')} to MNE volts")

    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(LIU_SOURCE_SFREQ)
    raw = mne.io.RawArray(continuous, info, verbose=False)

    raw = apply_mne_raw_pipeline(raw, config, runtime_steps)

    effective_sfreq = float(raw.info["sfreq"])
    data = mne_volts_to_model_unit(raw.get_data(), config)
    runtime_steps.append(f"convert MNE volts to model {config.get('final_model_unit')}")

    expected_samples_per_trial = int(round(rawdata.shape[2] * effective_sfreq / float(LIU_SOURCE_SFREQ)))
    total_expected = n_trials * expected_samples_per_trial
    if data.shape[1] != total_expected:  # type: ignore
        n_full = data.shape[1] // n_trials  # type: ignore
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]  # type: ignore
        runtime_steps.append(f"trim continuous samples to full trials: {expected_samples_per_trial} samples/trial")

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)  # type: ignore

    start_sample = int(round(float(config["mi_window_start_s"]) * effective_sfreq))
    window_samples = int(config["target_window_samples"]) if config.get("target_window_samples") is not None else int(round(float(config["target_window_s"]) * effective_sfreq))
    stop_sample = start_sample + window_samples

    if stop_sample > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop [{start_sample}:{stop_sample}] exceeds trial length "
            f"{X_rs.shape[-1]} at effective_sfreq={effective_sfreq}"
        )

    X_win = X_rs[:, :, start_sample:stop_sample]
    runtime_steps.append(f"crop fixed window samples [{start_sample}:{stop_sample}]")

    y = labels_to_zero_based(labels)
    X_win, y, reject_stats = maybe_reject_bad_trials(X_win, y, subject_id, config, runtime_steps)
    preprocessing_stats.update(reject_stats)

    # Save both the intended pipeline and the actual runtime steps.
    preprocessing_stats["pipeline_plan"] = steps
    preprocessing_stats["runtime_steps"] = runtime_steps

    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial), runtime_steps, preprocessing_stats


[2026-06-14 11:20:33] Configured preprocessing pipeline:
[2026-06-14 11:20:33]   - [ON] source::select_eeg_channels | - select Liu EEG channels, drop CPz reference, EOG, and marker before model input
[2026-06-14 11:20:33]   - [off] source::demean | mode=none | baseline_window_s=[0.0, 2.0]
[2026-06-14 11:20:33]   - [off] source::detrend | mode=none
[2026-06-14 11:20:33]   - [off] source::eog_correction | mode=none
[2026-06-14 11:20:33]   - [off] source::artifact_clipping | mode=none | percentile=99.5
[2026-06-14 11:20:33]   - [ON] mne_raw::reference | timing=before_resample_filter | mode=average
[2026-06-14 11:20:33]   - [ON] mne_raw::resample | sfreq=128
[2026-06-14 11:20:33]   - [ON] mne_raw::bandpass_filter | l_freq=0.5 | h_freq=40.0 | method=fir | phase=zero | fir_design=firwin
[2026-06-14 11:20:33]   - [off] mne_raw::notch_filter | timing=after_bandpass
[2026-06-14 11:20:33]   - [ON] window::crop_fixed_mi_window | start_s=1.5 | target_window_samples=537
[2026-06-14 11:20:33]   - [o

### 3.3 Dataset classes

In [10]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)
        if self.X.ndim != 3:
            raise ValueError(f"SubjectArrayDataset expects X as N x C x T, got shape={self.X.shape}.")
        if len(self.X) != len(self.y):
            raise ValueError(f"X/y length mismatch: {len(self.X)} windows vs {len(self.y)} labels.")

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected one EEG window as C x T, got shape={x.shape}.")
        return x, int(self.y[idx])


class FoldNormalizedDataset(Dataset):
    """Wrap a dataset and apply either fold-fitted or trial-wise normalization."""

    def __init__(self, dataset, normalizer_state):
        self.dataset = dataset
        self.normalizer_state = normalizer_state or {"mode": "none"}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = apply_normalizer_to_array(x, self.normalizer_state)
        if x.ndim != 2:
            raise ValueError(f"Normalization must return C x T, got shape={x.shape}.")
        return x.astype(np.float32), int(y)


### 3.4 JSON-safe helpers

In [11]:
def _json_safe_float(value, decimals=8):
    if value is None:
        return None
    value = float(np.nan_to_num(value, nan=0.0, posinf=0.0, neginf=0.0))
    return round(value, decimals)

def _json_safe_float_list(values, decimals=8):
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.round(arr, decimals=decimals).tolist()


### 3.5 Pretrained S-JEPA model builder (for embedding extraction)

In [12]:
NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")

def build_model():
    common_kwargs = {
        "n_chans": len(CH_NAMES),
        "chs_info": CHS_INFO,
        "n_times": WINDOW_SAMPLES,
        "n_outputs": TARGET_N_CLASSES,
    }
    mode = CONFIG["pretrained_mode"]
    if mode == "from_pretrained":
        model = SignalJEPA_PreLocal.from_pretrained(
            CONFIG["pretrained_repo_id"],
            **common_kwargs,
            strict=False,
        )
        info = {
            "loading_path": "from_pretrained",
            "repo_id": CONFIG["pretrained_repo_id"],
            "mode": mode,
        }
    elif mode == "random":
        model = SignalJEPA_PreLocal(**common_kwargs)
        info = {
            "loading_path": "random_initialization",
            "repo_id": None,
            "mode": mode,
        }
    else:
        raise ValueError("pretrained_mode must be 'from_pretrained' or 'random'.")
    info["model_name"] = CONFIG["model_name"]
    return model, info

def set_trainable_params_for_phase(model, phase):
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")

    if phase == "full":
        for _, p in model.named_parameters():
            p.requires_grad = True
        phase_groups = ["all_parameters"]
    else:
        for _, p in model.named_parameters():
            p.requires_grad = False
        for name, p in model.named_parameters():
            if any(name.startswith(prefix) for prefix in NEW_LAYER_PREFIXES):
                p.requires_grad = True
        phase_groups = list(NEW_LAYER_PREFIXES)

    trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable == 0:
        raise RuntimeError(f"No trainable parameters for phase={phase}.")
    return {
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def summarize_trainable_parameters(model):
    rows = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            rows.append({
                "name": name,
                "numel": int(param.numel()),
                "shape": list(param.shape),
            })
    return rows

def count_trainable_from_rows(rows):
    return int(sum(row.get("numel", 0) for row in rows))


### 3.6 Data driver — builds X_ALL / Y_ALL / SUBJECT_ID_ALL / CH_NAMES / CHS_INFO

In [13]:
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
MAT_FILES = []

files = find_source_mat_files(SOURCE_EXTRACT_DIR)

if files:
    MAT_FILES = files

if not MAT_FILES:
    raise FileNotFoundError(
        "Could not find Liu2024 source .mat files. "
    )

print(f"Source extract dir: {SOURCE_EXTRACT_DIR}")
print(f"Found {len(MAT_FILES)} .mat files")

# Write a structure preview for the first subject. This makes it clear whether
# the local files expose top-level rawdata/labels or an `eeg` struct.
preview_path = ARTIFACT_DIR / "mat_structure_preview_first_subject.csv"
mat_structure_preview(MAT_FILES[0]).to_csv(preview_path, index=False)
print(f"MAT structure preview saved to: {preview_path}")

subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if CONFIG["subjects_to_use"] is not None and sid not in set(int(s) for s in CONFIG["subjects_to_use"]):
        continue
    if sid in set(int(s) for s in CONFIG["exclude_subjects"]):
        continue
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    validate_liu_source_subject(X_raw, y_raw, sid, path=p)
    subjects.append({
        "subject_id": sid,
        "path": str(p),
        "rawdata_shape": tuple(X_raw.shape),
        "labels_shape": tuple(y_raw.shape),
        "label_counts_raw": np.bincount(y_raw.astype(int), minlength=3).tolist(),
        "raw_field": raw_field,
        "label_field": label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values("subject_id").reset_index(drop=True)
if subjects_df.empty:
    raise RuntimeError("No subjects loaded.")

SUBJECTS = [int(s) for s in subjects_df["subject_id"].tolist()]
print(f"Subjects loaded: {SUBJECTS}")

subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
subjects_df.to_csv(subject_inventory_path, index=False)
print(f"Subject inventory saved to: {subject_inventory_path}")

EEG_INFO = make_liu_info(EFFECTIVE_SFREQ)
CHS_INFO = EEG_INFO["chs"]
CH_NAMES = list(EEG_CHANNEL_NAMES)

Xs, ys, subject_ids = [], [], []
window_summary_rows = []
preprocessing_steps_first_subject = None

for item in subjects_df.to_dict("records"):
    sid = int(item["subject_id"])
    X_raw, y_raw, _, _ = load_subject_mat(Path(item["path"]))
    X_win, y, samples_per_trial, preprocessing_steps, preprocessing_stats = preprocess_subject_configurable(X_raw, y_raw, sid)

    if preprocessing_steps_first_subject is None:
        preprocessing_steps_first_subject = preprocessing_steps

    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))
    window_summary_rows.append({
        "subject_id": sid,
        "n_windows": int(len(y)),
        "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
        "preprocessed_shape": tuple(X_win.shape),
        "resampled_samples_per_trial": int(samples_per_trial),
        "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
        "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
        "target_window_samples": int(WINDOW_SAMPLES),
        "effective_window_duration_s": float(TARGET_TRIAL_DURATION_S),
        "source_unit": CONFIG["source_unit"],
        "final_model_unit": CONFIG["final_model_unit"],
        "demean_mode": CONFIG["demean_mode"],
        "reference_mode": CONFIG["reference_mode"],
        "reference_timing": CONFIG["reference_timing"],
        "resample": bool(CONFIG["resample"]),
        "effective_sfreq": float(EFFECTIVE_SFREQ),
        "filter_enabled": bool(CONFIG["filter_enabled"]),
        "filter_low": CONFIG.get("filter_low"),
        "filter_high": CONFIG.get("filter_high"),
        "filter_method": CONFIG.get("filter_method"),
        "mi_window_start_s": float(CONFIG["mi_window_start_s"]),
        "eog_correction": CONFIG.get("eog_correction"),
        "artifact_clip_mode": CONFIG.get("artifact_clip_mode"),
        "reject_bad_trials": bool(CONFIG.get("reject_bad_trials", False)),
        "normalization_mode": CONFIG.get("normalization_mode"),
        **preprocessing_stats,
    })

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0)
SUBJECT_ID_ALL = np.asarray(subject_ids)
print(f"X_ALL shape: {X_ALL.shape} | Y_ALL counts: {np.bincount(Y_ALL).tolist()}")

if preprocessing_steps_first_subject is not None:
    print("Preprocessing steps used:")
    for step in preprocessing_steps_first_subject:
        print(f"  - {step}")

window_summary_df = pd.DataFrame(window_summary_rows).sort_values("subject_id")
window_summary_path = ARTIFACT_DIR / "window_counts_by_subject.csv"
window_summary_df.to_csv(window_summary_path, index=False)
print(f"Window summary saved to: {window_summary_path}")
display(window_summary_df.head())

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

SUBJECT_WINDOWS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    idx = np.where(SUBJECT_ID_ALL == sid)[0]
    SUBJECT_WINDOWS[str(sid)] = SubjectArrayDataset(X_ALL[idx], Y_ALL[idx], subject_id=sid)


[2026-06-14 11:20:33] Source extract dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_data/liu2024_figshare/sourcedata
[2026-06-14 11:20:33] Found 50 .mat files
[2026-06-14 11:20:34] MAT structure preview saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-sjepa-riemann-crosssubject-fusion/20260614_1120_82d1a0d1/mat_structure_preview_first_subject.csv
[2026-06-14 11:20:40] Subjects loaded: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
[2026-06-14 11:20:40] Subject inventory saved to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-sjepa-riemann-crosssubject-fusion/20260614_1120_82d1a0d1/subject_inventory.csv
[2026-06-14 11:20:54] X_ALL shape: (2000, 29, 537) | Y_ALL counts: [1000, 1000]
[2026

,subject_id,n_windows,class_counts,preprocessed_shape,resampled_samples_per_trial,crop_start_sample,crop_stop_sample,target_window_samples,effective_window_duration_s,source_unit,...,reject_bad_trials,normalization_mode,artifact_clip_applied,artifact_clip_threshold,n_trials_before_reject,n_trials_after_reject,n_rejected_trials,rejection_skipped,pipeline_plan,runtime_steps
0,1,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
1,2,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
2,3,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
3,4,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...
4,5,40,"[20, 20]","(40, 29, 537)",1024,192,729,537,4.195312,microvolts,...,False,none,False,None,40,40,0,False,[[ON] source::select_eeg_channels | - select L...,[select 29 EEG channels; drop CPz source refer...


# 4. Euclidean Alignment and the two feature branches

EA whitens each subject by their reference covariance so pooled cross-subject training is
meaningful. The Riemannian branch maps aligned covariances to tangent-space vectors; the S-JEPA
branch extracts frozen-encoder embeddings (input to `final_layer`, pooled).

In [14]:
FUS = CONFIG["fusion"]

def _inv_sqrt_spd(R, eps=1e-6):
    w, V = eigh(R); w = np.clip(w, eps, None)
    return (V * (1.0 / np.sqrt(w))) @ V.T

def _logm_spd(C, eps=1e-12):
    w, V = eigh(C); w = np.clip(w, eps, None)
    return (V * np.log(w)) @ V.T

def euclidean_align(X):
    """X: (n, ch, t) for ONE subject -> EA-aligned (n, ch, t). Unsupervised (no labels)."""
    covs = np.einsum("nct,ndt->ncd", X, X) / X.shape[2]
    P = _inv_sqrt_spd(covs.mean(0))
    return np.einsum("cd,ndt->nct", P, X)

def trial_covariances(X):
    if HAVE_PYRIEMANN:
        return Covariances(estimator=FUS["cov_estimator"]).transform(X)
    n, c, t = X.shape
    out = np.empty((n, c, c))
    for i in range(n):
        Xi = X[i] - X[i].mean(1, keepdims=True)
        C = Xi @ Xi.T / (t - 1)
        out[i] = C + 1e-3 * (np.trace(C) / c) * np.eye(c)
    return out

def tangent_fit_transform(cov_train, cov_test):
    """Fold-safe tangent-space vectors. pyriemann affine-invariant, else log-Euclidean."""
    if HAVE_PYRIEMANN:
        ts = TangentSpace(metric=FUS["tangent_metric"]).fit(cov_train)
        return ts.transform(cov_train), ts.transform(cov_test)
    iu = np.triu_indices(cov_train.shape[1])
    def vec(C):
        L = _logm_spd(C); return L[iu]
    return np.array([vec(C) for C in cov_train]), np.array([vec(C) for C in cov_test])

def get_submodule_by_suffix(model, name):
    for n, m in model.named_modules():
        if n == name or n.endswith("." + name):
            return m
    return None

def extract_embeddings(model, X, device, module_name, pool, batch):
    """Capture the input to `module_name` (e.g. final_layer) as the trial embedding."""
    model.eval().to(device)
    target = get_submodule_by_suffix(model, module_name)
    if target is None:
        raise RuntimeError(f"submodule '{module_name}' not found; set CONFIG['fusion']['embedding_module']")
    store = {}
    h = target.register_forward_pre_hook(lambda m, inp: store.__setitem__("z", inp[0].detach()))
    out = []
    try:
        with torch.no_grad():
            for i in range(0, len(X), batch):
                xb = torch.as_tensor(np.asarray(X[i:i + batch]), dtype=torch.float32, device=device)
                model(xb)
                z = store["z"]
                if z.ndim == 3:
                    z = z.mean(1) if pool == "mean" else z.reshape(z.shape[0], -1)
                elif z.ndim > 3:
                    z = z.reshape(z.shape[0], -1)
                out.append(z.float().cpu().numpy())
    finally:
        h.remove()
    return np.concatenate(out, 0)


# 5. Build aligned per-trial arrays for both branches

For each subject: EA-align the windows, compute covariances (Riemannian branch), and extract
S-JEPA embeddings (once, cached). Trials stay in `X_ALL` order so everything lines up.

In [15]:
subjects_all = sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key)
SUBJ = np.array([str(s) for s in SUBJECT_ID_ALL])
Y = Y_ALL.astype(int)

# 5a. EA-aligned windows per subject (concatenated back into X_ALL order)
X_aligned = np.empty_like(X_ALL, dtype=np.float64)
for sid in subjects_all:
    m = SUBJ == str(sid)
    Xs = X_ALL[m].astype(np.float64)
    X_aligned[m] = euclidean_align(Xs) if FUS["euclidean_alignment"] else Xs
print(f"EA applied: {FUS['euclidean_alignment']} | aligned array {X_aligned.shape}")

# 5b. Riemannian covariances on aligned windows
COV_ALL = trial_covariances(X_aligned)
print(f"Covariances: {COV_ALL.shape}")

# 5c. S-JEPA embeddings (cached). Priority: precomputed file -> live extraction -> disabled.
EMB_ALL, SJEPA_OK = None, False
emb_path = FUS.get("sjepa_embeddings_path")
if emb_path and Path(emb_path).exists():
    EMB_ALL = np.load(emb_path)["X"]; SJEPA_OK = EMB_ALL.shape[0] == len(Y)
    print(f"Loaded precomputed embeddings {EMB_ALL.shape} (aligned to X_ALL order): ok={SJEPA_OK}")
elif "sjepa" in FUS["branches"] or "fusion" in FUS["branches"]:
    if HAVE_TORCH and HAVE_BRAINDECODE:
        try:
            model, info = build_model()
            src = X_aligned if FUS["align_before_sjepa"] else X_ALL.astype(np.float64)
            EMB_ALL = extract_embeddings(model, src, DEVICE, FUS["embedding_module"],
                                         FUS["embedding_pool"], FUS["embedding_batch"])
            SJEPA_OK = True
            np.savez(ARTIFACT_DIR / "sjepa_embeddings.npz", X=EMB_ALL)
            print(f"Extracted S-JEPA embeddings {EMB_ALL.shape}; cached to artifacts.")
        except Exception as exc:
            print(f"[S-JEPA] embedding extraction failed -> running Riemann-only. Reason: {exc}")
            print("         Fix CONFIG['fusion']['embedding_module'] or supply 'sjepa_embeddings_path'.")
    else:
        print("[S-JEPA] torch/braindecode unavailable and no precomputed embeddings -> Riemann-only.")

# resolve which branches are actually runnable
branches = [b for b in FUS["branches"]
            if not ((b in ("sjepa", "fusion")) and not SJEPA_OK)]
if branches != FUS["branches"]:
    print(f"Branches adjusted for availability: {branches}")


[2026-06-14 11:20:55] EA applied: True | aligned array (2000, 29, 537)
[2026-06-14 11:20:55] Covariances: (2000, 29, 29)


[2026-06-14 11:20:57] Extracted S-JEPA embeddings (2000, 64); cached to artifacts.


# 6. Cross-subject (LOSO) / within-subject evaluation with fusion

`block_features` assembles the standardized feature blocks for a branch given train/test trial
indices (tangent fit on train only; scalers fit on train only — fold-safe). `fit_predict_branch`
trains the linear head (or stacking meta-learner) and predicts.

In [16]:
def block_features(branch, tr, te):
    """Return (Xtr, Xte) for the requested branch, fold-safe."""
    blocks_tr, blocks_te = [], []
    if branch in ("riemann", "fusion"):
        ftr, fte = tangent_fit_transform(COV_ALL[tr], COV_ALL[te])
        blocks_tr.append(ftr); blocks_te.append(fte)
    if branch in ("sjepa", "fusion"):
        blocks_tr.append(EMB_ALL[tr]); blocks_te.append(EMB_ALL[te])
    Xtr = np.concatenate(blocks_tr, axis=1); Xte = np.concatenate(blocks_te, axis=1)
    if FUS["standardize"]:
        sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    return Xtr, Xte

def _logreg():
    return LogisticRegression(max_iter=2000, C=FUS["logreg_C"])

def fit_predict_branch(branch, tr, te, ytr):
    if branch == "fusion" and FUS["fusion_method"] == "stacking":
        # probability-level stacking: each sub-branch -> meta logistic regression (leak-free via CV)
        metas_tr, metas_te = [], []
        for sub in ("riemann", "sjepa"):
            Xtr, Xte = block_features(sub, tr, te)
            clf = _logreg()
            ptr = cross_val_predict(clf, Xtr, ytr, cv=3, method="predict_proba")[:, 1]
            clf.fit(Xtr, ytr); pte = clf.predict_proba(Xte)[:, 1]
            metas_tr.append(ptr); metas_te.append(pte)
        Mtr = np.column_stack(metas_tr); Mte = np.column_stack(metas_te)
        meta = _logreg().fit(Mtr, ytr)
        return meta.predict(Mte)
    Xtr, Xte = block_features(branch, tr, te)
    return _logreg().fit(Xtr, ytr).predict(Xte)

def evaluate_loso(branch, subj_subset):
    rows = []
    for held in subj_subset:
        te = np.where(SUBJ == str(held))[0]
        tr = np.where(SUBJ != str(held))[0]
        if len(te) == 0:
            continue
        yp = fit_predict_branch(branch, tr, te, Y[tr])
        rows.append({"branch": branch, "held_subject": str(held), "n_test": int(len(te)),
                     "balanced_accuracy": float(balanced_accuracy_score(Y[te], yp)),
                     "accuracy": float(accuracy_score(Y[te], yp))})
    return rows

def evaluate_within(branch, subj_subset):
    rows = []
    for sid in subj_subset:
        idx = np.where(SUBJ == str(sid))[0]; y = Y[idx]
        skf = StratifiedKFold(n_splits=FUS["within_cv_folds"], shuffle=True, random_state=FUS["seed"])
        bals = []
        for itr, ite in skf.split(idx, y):
            tr, te = idx[itr], idx[ite]
            yp = fit_predict_branch(branch, tr, te, Y[tr])
            bals.append(balanced_accuracy_score(Y[te], yp))
        rows.append({"branch": branch, "held_subject": str(sid), "n_test": int(len(idx)),
                     "balanced_accuracy": float(np.mean(bals)), "accuracy": None})
    return rows


# 7. Run and compare branches

In [17]:
subset = (subjects_all if FUS["decodable_subjects"] == "all"
          else [str(s) for s in FUS["decodable_subjects"] if str(s) in set(SUBJ)])
print(f"Evaluating on {len(subset)} subjects ({FUS['eval_mode']}): {subset}\n")

runner = evaluate_loso if FUS["eval_mode"] == "loso" else evaluate_within
ALL_ROWS = []
for branch in branches:
    rows = runner(branch, subset)
    ALL_ROWS.extend(rows)
    bals = [r["balanced_accuracy"] for r in rows]
    print(f"  {branch:>8}: mean balanced accuracy = {100*np.mean(bals):.1f}%  (sd {100*np.std(bals):.1f}, n={len(bals)})")

RESULT_DF = pd.DataFrame(ALL_ROWS)
RESULT_DF.to_csv(ARTIFACT_DIR / "crosssubject_fusion_results.csv", index=False)

summary = {}
for branch in branches:
    b = RESULT_DF[RESULT_DF["branch"] == branch]["balanced_accuracy"]
    summary[branch] = {"mean_balanced_accuracy": float(b.mean()), "std": float(b.std()), "n": int(len(b))}
summary_meta = {"eval_mode": FUS["eval_mode"], "euclidean_alignment": FUS["euclidean_alignment"],
                "fusion_method": FUS["fusion_method"], "sjepa_available": bool(SJEPA_OK),
                "n_subjects_eval": len(subset), "branch_summary": summary}
with open(ARTIFACT_DIR / "crosssubject_fusion_summary.json", "w") as f:
    json.dump(summary_meta, f, indent=2, default=str)

print("\n================ SUMMARY (balanced accuracy on the evaluation subgroup) ================")
for branch in branches:
    print(f"  {branch:>8}: {100*summary[branch]['mean_balanced_accuracy']:.1f}%")
if "fusion" in summary and "riemann" in summary:
    delta = 100 * (summary["fusion"]["mean_balanced_accuracy"] - summary["riemann"]["mean_balanced_accuracy"])
    print(f"\n  fusion - riemann = {delta:+.1f} points  (does S-JEPA add anything over Riemannian alone?)")


[2026-06-14 11:20:57] Evaluating on 10 subjects (loso): ['7', '20', '23', '26', '28', '37', '38', '40', '44', '45']

[2026-06-14 11:21:08]    riemann: mean balanced accuracy = 52.0%  (sd 4.4, n=10)
[2026-06-14 11:21:08]      sjepa: mean balanced accuracy = 50.0%  (sd 6.8, n=10)
[2026-06-14 11:21:19]     fusion: mean balanced accuracy = 52.8%  (sd 4.8, n=10)

[2026-06-14 11:21:19] ================ SUMMARY (balanced accuracy on the evaluation subgroup) ================
[2026-06-14 11:21:19]    riemann: 52.0%
[2026-06-14 11:21:19]      sjepa: 50.0%
[2026-06-14 11:21:19]     fusion: 52.8%

[2026-06-14 11:21:19]   fusion - riemann = +0.8 points  (does S-JEPA add anything over Riemannian alone?)


# 8. Plots

In [18]:
if HAVE_MPL and not RESULT_DF.empty:
    piv = RESULT_DF.pivot_table(index="held_subject", columns="branch", values="balanced_accuracy")
    piv = piv.reindex([s for s in subset if s in piv.index])
    x = np.arange(len(piv)); w = 0.8 / max(len(branches), 1)
    fig, ax = plt.subplots(figsize=(max(8, 0.5 * len(piv)), 4.5))
    for k, branch in enumerate(branches):
        if branch in piv.columns:
            ax.bar(x + k * w, piv[branch].values, width=w, label=branch)
    ax.axhline(0.5, ls="--", c="grey", lw=1, label="chance")
    ax.axhline(0.7, ls=":", c="green", lw=1, label="0.70 target")
    ax.set_xticks(x + 0.4 - w / 2); ax.set_xticklabels(piv.index, rotation=0)
    ax.set_ylabel("balanced accuracy"); ax.set_ylim(0, 1)
    ax.set_title(f"Cross-subject fusion ({FUS['eval_mode']}) — per held-out subject")
    ax.legend(fontsize=8, ncol=2); fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "crosssubject_fusion_per_subject.png", dpi=160); plt.close(fig)
    print(f"Saved: {ARTIFACT_DIR/'crosssubject_fusion_per_subject.png'}")

    fig, ax = plt.subplots(figsize=(5, 4))
    means = [summary[b]["mean_balanced_accuracy"] for b in branches]
    sds = [summary[b]["std"] for b in branches]
    ax.bar(branches, means, yerr=sds, capsize=4)
    ax.axhline(0.5, ls="--", c="grey", lw=1); ax.axhline(0.7, ls=":", c="green", lw=1)
    ax.set_ylim(0, 1); ax.set_ylabel("mean balanced accuracy")
    ax.set_title("Branch comparison"); fig.tight_layout()
    fig.savefig(ARTIFACT_DIR / "crosssubject_fusion_branch_means.png", dpi=160); plt.close(fig)
    print(f"Saved: {ARTIFACT_DIR/'crosssubject_fusion_branch_means.png'}")
else:
    print("Plots skipped.")


[2026-06-14 11:21:19] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-sjepa-riemann-crosssubject-fusion/20260614_1120_82d1a0d1/crosssubject_fusion_per_subject.png
[2026-06-14 11:21:19] Saved: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-sjepa-riemann-crosssubject-fusion/20260614_1120_82d1a0d1/crosssubject_fusion_branch_means.png


## 9. Save run metadata

In [19]:
run_metadata = {
    "run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR),
    "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"],
    "fusion_config": FUS, "branches_run": branches, "sjepa_available": bool(SJEPA_OK),
    "tangent_backend": "pyriemann" if HAVE_PYRIEMANN else "log_euclidean_fallback",
    "summary": summary_meta,
}
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)
print(f"Saved run metadata to: {ARTIFACT_DIR/'run_metadata.json'}")
for p in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  - {p.name}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass


[2026-06-14 11:21:19] Saved run metadata to: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-sjepa-riemann-crosssubject-fusion/20260614_1120_82d1a0d1/run_metadata.json
[2026-06-14 11:21:19]   - config.json
[2026-06-14 11:21:19]   - crosssubject_fusion_branch_means.png
[2026-06-14 11:21:19]   - crosssubject_fusion_per_subject.png
[2026-06-14 11:21:19]   - crosssubject_fusion_results.csv
[2026-06-14 11:21:19]   - crosssubject_fusion_summary.json
[2026-06-14 11:21:19]   - mat_structure_preview_first_subject.csv
[2026-06-14 11:21:19]   - run.log
[2026-06-14 11:21:19]   - run_metadata.json
[2026-06-14 11:21:19]   - sjepa_embeddings.npz
[2026-06-14 11:21:19]   - subject_inventory.csv
[2026-06-14 11:21:19]   - window_counts_by_subject.csv


# 10. How to read this, and how to push toward 70%

- **Branch comparison is the headline.** `riemann` is the strong baseline; `sjepa` is the frozen
  encoder alone; `fusion` is both. If `fusion` clearly beats `riemann`, S-JEPA is contributing
  complementary signal — the result you want. If not, the embedding isn't adding value yet, which
  is itself a clean, reportable finding.
- **EA + LOSO is the lever.** Compare `euclidean_alignment: true` vs `false`: alignment should
  lift cross-subject accuracy noticeably. LOSO gives every model hundreds–thousands of training
  trials, which is the regime S-JEPA needs.
- **Knobs to try toward 70%** (each is one CONFIG edit): switch `fusion_method` to `stacking`;
  raise/lower `logreg_C`; set `decodable_subjects` to `"all"` vs the curated subgroup; toggle
  `align_before_sjepa`; try `eval_mode: within` for a per-subject comparison. The highest-yield
  single change is usually keeping EA on and evaluating on the decodable subgroup.
- **If S-JEPA stays at chance even pooled**, freeze it and treat the embedding as one block in the
  fusion; the Riemannian block will carry the subgroup to its ceiling, and the comparison tells
  you precisely how much (if anything) the SSL representation adds.

**Validated vs environment-dependent.** EA, covariance/tangent features, fusion, stacking, and
the LOSO/within harness are validated. The S-JEPA embedding extraction depends on your
braindecode build and the pretrained checkpoint; if the live hook fails, point
`CONFIG["fusion"]["sjepa_embeddings_path"]` at an `.npz` (key `X`, shape `(n_trials, d)` in
`X_ALL` order) and the rest runs unchanged.